# Autoencoder Example 2 — Denoising Autoencoder

A small twist on the basic autoencoder that turns it into a practical tool: feed in a
**corrupted** image but ask it to reconstruct the **clean** original. The network learns
to *remove noise*.

## The one idea

A vanilla autoencoder is trained on `(x, x)` — input and target are identical. A
**denoising** autoencoder is trained on `(x + noise, x)`:

- **Input**: the image with random Gaussian noise added.
- **Target**: the original clean image.

Because the bottleneck can't pass the noise through faithfully, the cheapest way to lower
the loss is to learn the *underlying digit* and redraw it cleanly. This also makes the
learned features more robust — a classic regularisation trick (Vincent et al., 2008).

> Tested with `tensorflow==2.1`. Trains in seconds on CPU.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

np.random.seed(0); tf.random.set_seed(0)
print('TensorFlow', tf.__version__)

## 1. Data + add noise

Scale to `[0, 1]`, flatten, then make a **noisy copy** by adding Gaussian noise and
clipping back to `[0, 1]`. We keep both the clean and noisy versions.

In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32').reshape(-1, 784) / 255.0
x_test = x_test.astype('float32').reshape(-1, 784) / 255.0

NOISE = 0.5
x_train_noisy = np.clip(x_train + NOISE * np.random.normal(size=x_train.shape), 0, 1).astype('float32')
x_test_noisy = np.clip(x_test + NOISE * np.random.normal(size=x_test.shape), 0, 1).astype('float32')
print('Train:', x_train.shape)

## 2. Model

Same dense encoder/decoder shape as the basic autoencoder (`784→128→64→32→...→784`). The
architecture doesn't change — only **what we feed vs. what we ask for** does.

In [ ]:
autoencoder = tf.keras.Sequential([
    layers.Input((784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),    # bottleneck
    layers.Dense(64, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(784, activation='sigmoid'),
])
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder.summary()

## 3. Train

The crucial line: input is **`x_train_noisy`**, target is **`x_train`** (clean).

In [ ]:
autoencoder.fit(x_train_noisy, x_train,
                epochs=20, batch_size=256, shuffle=True,
                validation_data=(x_test_noisy, x_test), verbose=2)

## 4. Noisy input → denoised output

Three rows: the noisy input we feed, the model's denoised output, and the clean target.

In [ ]:
n = 10
denoised = autoencoder.predict(x_test_noisy[:n])
rows = [(x_test_noisy, 'noisy input'), (denoised, 'denoised'), (x_test, 'clean target')]
plt.figure(figsize=(n * 1.5, 4.5))
for r, (data, title) in enumerate(rows):
    for i in range(n):
        ax = plt.subplot(3, n, r * n + i + 1)
        plt.imshow(np.asarray(data)[i].reshape(28, 28), cmap='gray'); plt.axis('off')
        if i == 0: ax.set_title(title, loc='left')
plt.show()

### Things to try

- Crank `NOISE` up to 1.0 — how much can it still recover?
- Try other corruptions: salt-and-pepper noise, or randomly masking pixels (dropout).
- Make it convolutional (see `convolutional_autoencoder.ipynb`) for sharper denoising.
- Use the trained encoder's features as input to a classifier.